# Salary Prediction

**Regression Project 54**

Full pipeline: EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.

## 2. Project Overview

Predict data scientist salaries using job listing attributes from Glassdoor. The dataset contains
job titles, company ratings, locations, company size, industry, sector, and salary estimates.

We parse the salary range into a numeric midpoint and predict it from job and company features.
This project teaches text parsing, high-cardinality feature handling, and real-world salary modeling.

## 3. Learning Objectives

1. Parse salary ranges from text to numeric targets
2. Handle high-cardinality categorical features (Job Title, Location, Industry)
3. Extract meaningful features from messy real-world text data
4. Understand salary drivers in the data science job market
5. Compare baseline, LazyPredict, FLAML, and top-3 models
6. Evaluate with R², RMSE, MAE

## 4. Problem Statement

> Given job listing attributes from Glassdoor, predict the **midpoint salary** (in $K).

This is a **regression** task. Primary metric: **R²**; secondary: RMSE, MAE.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Job seekers | Negotiate fair compensation |
| HR / Recruiters | Benchmark competitive offers |
| Companies | Compensation planning and budgeting |
| Career coaches | Data-driven career advice |
| ML learners | Real-world messy data regression |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Source | Kaggle: `rashikrahmanpritom/data-science-job-posting-on-glassdoor` |
| Records | ~956 |
| Features | Job Title, Rating, Company, Location, Size, Industry, Sector, Revenue, etc. |
| Target | `avg_salary` (midpoint of Salary Estimate range, in $K) |
| Key insight | Location, company size, and seniority level drive salary |

## 7. Dataset Source and License Notes

- **Kaggle**: [rashikrahmanpritom/data-science-job-posting-on-glassdoor](https://www.kaggle.com/datasets/rashikrahmanpritom/data-science-job-posting-on-glassdoor)
- Scraped from Glassdoor job listings
- License: check Kaggle dataset page
- Downloaded via `kagglehub` at runtime

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ['lazypredict', 'flaml', 'kagglehub', 'xgboost', 'lightgbm']:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Environment ready.')

## 9. Imports

In [ ]:
import os, warnings, glob, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
print('All imports loaded.')

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = 'rashikrahmanpritom/data-science-job-posting-on-glassdoor'
TARGET = 'avg_salary'
TEST_SIZE = 0.15
VAL_SIZE = 0.15
FLAML_BUDGET = 90
MAX_ROWS = 50000

## 11. Dataset Download or Loading

Download the Glassdoor Data Science jobs dataset via `kagglehub`.

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f'Downloaded to: {path}')
except Exception as e:
    raise RuntimeError(
        f'Dataset download failed: {e}\n'
        'Ensure KAGGLE_API_TOKEN is set. '
        'Fallback: download from https://www.kaggle.com/datasets/rashikrahmanpritom/data-science-job-posting-on-glassdoor'
    )
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV files found in {path}'
df_raw = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
print(f'Shape: {df_raw.shape}')
df_raw.head()

## 12. Data Validation Checks

We need to parse the `Salary Estimate` field from text (e.g., '$53K-$91K (Glassdoor est.)') into
a numeric midpoint. We also clean Company Name and handle missing values.

In [ ]:
# Parse salary: extract low/high from text range, compute midpoint
def parse_salary(s):
    """Extract salary midpoint in $K from Glassdoor format."""
    if pd.isna(s) or s == '-1':
        return np.nan
    nums = re.findall(r'\$(\d+)K', str(s))
    if len(nums) >= 2:
        return (int(nums[0]) + int(nums[1])) / 2
    elif len(nums) == 1:
        return float(nums[0])
    return np.nan

df = df_raw.copy()
if 'Salary Estimate' in df.columns:
    df[TARGET] = df['Salary Estimate'].apply(parse_salary)
    df = df.drop(columns=['Salary Estimate'])

# Drop rows without valid salary
df = df.dropna(subset=[TARGET])

# Clean company name (remove rating suffix)
if 'Company Name' in df.columns:
    df['Company Name'] = df['Company Name'].str.replace(r'\n.*', '', regex=True).str.strip()

# Drop ID-like columns
id_cols = [c for c in df.columns if c.lower() in ['unnamed: 0', 'index']]
if id_cols: df = df.drop(columns=id_cols)

df = df.drop_duplicates().reset_index(drop=True)
print(f'Missing:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
print(f'\nShape: {df.shape}')
print(f'Target range: ${df[TARGET].min():.0f}K - ${df[TARGET].max():.0f}K')

## 13. Exploratory Data Analysis

Let's explore salary distributions and key feature relationships.

In [ ]:
df.describe()

In [ ]:
# Drop high-cardinality / text columns for EDA simplicity
drop_for_model = ['Job Description', 'Competitors']
drop_existing = [c for c in drop_for_model if c in df.columns]
if drop_existing:
    df = df.drop(columns=drop_existing)
    print(f'Dropped columns: {drop_existing}')
print(f'Remaining columns: {df.columns.tolist()}')

In [ ]:
cat_feats = df.select_dtypes(include=['object']).columns.tolist()
for col in cat_feats[:5]:
    print(f'{col}: {df[col].nunique()} unique values')

In [ ]:
# Salary by key categories
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ['Size', 'Type of ownership', 'Sector']):
    if col in df.columns:
        top_vals = df[col].value_counts().head(8).index
        sub = df[df[col].isin(top_vals)]
        sub.groupby(col)[TARGET].mean().sort_values(ascending=False).plot.barh(ax=ax, color='steelblue')
        ax.set_title(f'Mean Salary by {col}')
plt.tight_layout(); plt.show()

In [ ]:
num_cols_eda = df.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols_eda) >= 2:
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(df[num_cols_eda].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Correlation Heatmap')
    plt.tight_layout(); plt.show()

## 14. Target Analysis

Data science salaries should roughly follow a normal distribution with a slight right skew.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET], bins=30, color='coral', alpha=0.7)
axes[0].set_title('Salary Distribution ($K)')
axes[1].boxplot(df[TARGET].dropna(), vert=True)
axes[1].set_title('Salary Box Plot')
plt.tight_layout(); plt.show()
print(f'Mean: ${df[TARGET].mean():.0f}K')
print(f'Median: ${df[TARGET].median():.0f}K')
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15. Train / Validation / Test Split

70/15/15 split with fixed seed.

In [ ]:
# Limit categorical cardinality: keep top N categories, group rest as 'Other'
def cap_cardinality(d, col, top_n=15):
    if col in d.columns and d[col].dtype == object:
        top_cats = d[col].value_counts().head(top_n).index
        d[col] = d[col].where(d[col].isin(top_cats), 'Other')
    return d

high_card = ['Job Title', 'Company Name', 'Location', 'Headquarters', 'Industry', 'Sector']
for col in high_card:
    df = cap_cardinality(df, col, top_n=15)

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 16. Preprocessing Strategy

- **Numeric**: Impute median + StandardScaler
- **Categorical**: Impute mode + OneHotEncoder (with cardinality already capped)
- High-cardinality columns have been capped to top-15 categories + 'Other'

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
transformers = []
if num_cols:
    transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}')

## 17. Feature Engineering

- **Seniority level**: Extract from Job Title (Senior, Junior, Lead, etc.)
- **State**: Extract from Location
- **Company age**: Current year minus Founded year

In [ ]:
def engineer_features(d):
    d = d.copy()
    if 'Job Title' in d.columns:
        title_lower = d['Job Title'].str.lower().fillna('')
        d['is_senior'] = title_lower.str.contains('senior|sr|lead|principal|director').astype(int)
        d['is_junior'] = title_lower.str.contains('junior|jr|entry|intern|associate').astype(int)
    if 'Location' in d.columns:
        d['state'] = d['Location'].str.split(',').str[-1].str.strip()
    if 'Founded' in d.columns:
        d['company_age'] = 2024 - pd.to_numeric(d['Founded'], errors='coerce')
        d['company_age'] = d['company_age'].clip(lower=0)
    return d

X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)
# Rebuild preprocessor
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
# Cap new cat cols cardinality
for col in ['state']:
    for d in [X_train, X_val, X_test]:
        if col in d.columns:
            top_cats = X_train[col].value_counts().head(15).index
            d[col] = d[col].where(d[col].isin(top_cats), 'Other')
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
transformers = []
if num_cols:
    transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Features after engineering: {X_train.shape[1]}')

## 18. Baseline Model

Three baselines: DummyRegressor (mean), LinearRegression, RandomForest.

In [ ]:
results = {}
for name, reg in [
    ('Dummy (mean)', DummyRegressor(strategy='mean')),
    ('LinearRegression', LinearRegression()),
    ('RandomForest', RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1))
]:
    pipe = Pipeline([('pre', preprocessor), ('reg', reg)])
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_val)
    r = {'RMSE': np.sqrt(mean_squared_error(y_val, yp)),
         'MAE': mean_absolute_error(y_val, yp),
         'R2': r2_score(y_val, yp)}
    results[name] = r
    print(f"{name}: R2={r['R2']:.4f}, RMSE=${r['RMSE']:.1f}K, MAE=${r['MAE']:.1f}K")

## 19. LazyPredict Benchmark

Quick comparison of ~30 regression models.

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

FLAML with 90-second budget.

In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task='regression', metric='r2',
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f'Best estimator: {automl.best_estimator}')
yp_fl = automl.predict(X_val_lp)
results['FLAML'] = {'RMSE': np.sqrt(mean_squared_error(y_val, yp_fl)),
                    'MAE': mean_absolute_error(y_val, yp_fl),
                    'R2': r2_score(y_val, yp_fl)}
print(f"FLAML: R2={results['FLAML']['R2']:.4f}, RMSE=${results['FLAML']['RMSE']:.1f}K")

## 21. Top 3 Model Selection

Based on benchmark results, we select three strong models.

In [ ]:
try:
    from lightgbm import LGBMRegressor; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBRegressor; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm:
    top3['LightGBM'] = LGBMRegressor(n_estimators=400, learning_rate=0.05, max_depth=5, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesRegressor
    top3['ExtraTrees'] = ExtraTreesRegressor(n_estimators=400, random_state=SEED, n_jobs=-1)
if has_xgb:
    top3['XGBoost'] = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=5, random_state=SEED, verbosity=0, n_jobs=-1)
else:
    from sklearn.ensemble import AdaBoostRegressor
    top3['AdaBoost'] = AdaBoostRegressor(n_estimators=200, random_state=SEED)
top3['GradBoosting'] = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=SEED)
print(f'Top 3: {list(top3.keys())}')

## 22. Final Training and Evaluation of Top 3

Train on training set, evaluate on held-out test set.

In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
final = {}
predictions = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t)
    predictions[name] = yp
    final[name] = {'RMSE': np.sqrt(mean_squared_error(y_test, yp)),
                   'MAE': mean_absolute_error(y_test, yp),
                   'R2': r2_score(y_test, yp)}
    print(f"{name}: R2={final[name]['R2']:.4f}, RMSE=${final[name]['RMSE']:.1f}K, MAE=${final[name]['MAE']:.1f}K")
pd.DataFrame(final).T.sort_values('R2', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    ax.scatter(y_test, yp, alpha=0.5, s=20)
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=2)
    ax.set_xlabel('Actual ($K)'); ax.set_ylabel('Predicted ($K)')
    ax.set_title(f"{name} (R2={final[name]['R2']:.3f})")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    residuals = y_test.values - yp
    ax.scatter(yp, residuals, alpha=0.5, s=20)
    ax.axhline(0, color='red', linestyle='--')
    ax.set_xlabel('Predicted ($K)'); ax.set_ylabel('Residual')
    ax.set_title(f'{name} Residuals')
plt.tight_layout(); plt.show()

## 23. Error Analysis

Where does the best model struggle most?

In [ ]:
best_name = max(final, key=lambda k: final[k]['R2'])
best_preds = predictions[best_name]
abs_errors = np.abs(y_test.values - best_preds)
print(f'Best model: {best_name}')
print(f'Mean absolute error: ${abs_errors.mean():.1f}K')
print(f'Median absolute error: ${np.median(abs_errors):.1f}K')
print(f'90th percentile error: ${np.percentile(abs_errors, 90):.1f}K')
print(f'Max error: ${abs_errors.max():.1f}K')

## 24. Interpretation and Business Insight

- **Seniority** (Senior/Lead titles) is the strongest salary driver
- **Location** matters significantly — Bay Area and NYC command premiums
- **Company size** and **industry sector** affect base compensation
- **Rating** has a mild positive correlation with salary
- **Revenue** of the company correlates with higher pay bands
- The Glassdoor salary range already smooths individual variation

## 25. Limitations

1. Target is a salary *estimate range*, not actual compensation
2. Only ~956 listings — limited sample
3. No years-of-experience or education features
4. Skills/tech-stack not extracted from job descriptions
5. Limited to data science roles — not generalizable to other fields

## 26. How to Improve This Project

1. NLP on Job Description to extract required skills/tools
2. More data from multiple job boards
3. Include years of experience as a feature
4. Target encoding for high-cardinality features
5. Cross-validation for more reliable estimates

## 27. Production Considerations

- Real-time salary estimator for job seekers
- Integration with job board APIs for continuous data
- Regional cost-of-living adjustment
- Model retraining as market conditions evolve
- Confidence intervals for salary estimates

## 28. Common Mistakes

1. Not parsing salary range from text
2. Keeping Job Description as raw text without extraction
3. One-hot encoding 500+ unique company names
4. Ignoring seniority signals in Job Title
5. Not handling -1 or missing values in Founded/Revenue

## 29. Mini Challenge / Exercises

1. Extract top 10 skills from Job Description using keyword matching
2. Build a model using only numeric features and compare
3. Try ordinal encoding for Size and Revenue columns
4. Compare performance with and without Location features
5. Try predicting salary range width (max - min) instead of midpoint

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Regression — predict data scientist salary |
| Dataset | Glassdoor DS jobs (~956 records) |
| Difficulty | Hard (noisy, high-cardinality features) |
| Key insight | Seniority, location, and company size drive salary |
| Best approach | Gradient boosting with engineered features |
| Primary metric | R² |
| Baseline comparison | Feature engineering significantly improves over raw-data models |